# Project 7 — Transformer From Scratch (ScratchFormer)

A complete experimental project: we build a **small encoder–decoder Transformer entirely from scratch in TensorFlow/Keras** — no `tf.keras.layers.MultiHeadAttention`, no Hugging Face — and train it on **English → French** translation. The notebook mirrors the 31-phase project blueprint:

`raw data → tokenizer → vocabulary → tensors → masks → embeddings → positional encoding → attention → multi-head attention → encoder → decoder → Transformer → training → autoregressive inference → attention visualization → ablation study → evaluation`

> Goal: *"I didn't just use Transformers — I understand the architecture well enough to implement, train, debug, inspect, experiment with, and evaluate one from the ground up."*

## 01 · Environment setup
Run in **Google Colab with GPU** (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
import os, sys
if 'COLAB_RELEASE_TAG' in os.environ and not os.path.exists('src'):
    !git clone https://github.com/Ravikishore710/ScratchFormer.git
    %cd ScratchFormer
!pip install -q -r requirements.txt

# make the repo root importable regardless of where the notebook lives
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.path.abspath('.'))
import tensorflow as tf
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 02 · Project configuration
A single `Config` object fully describes every run (seed, splits, architecture, optimizer).

In [ ]:
from src.config import Config
cfg = Config.load_json('configs/baseline.json')
cfg.validate()

import random, numpy as np
random.seed(cfg.seed); np.random.seed(cfg.seed); tf.random.set_seed(cfg.seed)
cfg.to_dict()

## 03–04 · Dataset (English → French, manythings fra-eng)
A real, small parallel corpus (~10 MB) that trains in minutes on Colab. We download it, parse the pairs, cap sequence lengths, and subsample 60k pairs.

In [ ]:
from src.data.dataset import download_fra_eng, load_pairs, prepare_datasets
txt = download_fra_eng(cfg)
pairs = load_pairs(txt)
print(f'available pairs: {len(pairs):,}')
for e, f in pairs[:5]:
    print(f'  EN: {e}')
    print(f'  FR: {f}\n')

## 05 · Train / validation / test split
80 / 10 / 10 with seed 42. **The test set stays untouched until final evaluation.**

In [ ]:
bundle = prepare_datasets(cfg, verbose=True)  # tokenizers fit on TRAIN only
print('\nexample batch shapes:')
for (src, dec_in), tgt in bundle.train_ds.take(1):
    print('  src:', src.shape, '| dec_in:', dec_in.shape, '| tgt:', tgt.shape)

## 06–08 · Tokenization, vocabulary, special tokens
Explicit `token → id` / `id → token` maps built **only from training data**. Special tokens: `<PAD>=0`, `<UNK>=1`, `<SOS>=2`, `<EOS>=3`.

In [ ]:
src_tok, tgt_tok = bundle.src_tok, bundle.tgt_tok
print('src vocab:', len(src_tok), '| tgt vocab:', len(tgt_tok))
text = 'I am a student.'
ids = src_tok.encode(text, add_special=True, max_len=cfg.max_seq_len)
print('tokens :', src_tok.tokenize(text))
print('ids    :', ids)
print('decoded:', src_tok.decode(ids))

## 09 · `tf.data` pipeline + teacher forcing
Decoder input is the target shifted right by one (`<SOS> A B C D`), the target is `A B C D <EOS>` — that is teacher forcing. Batching, shuffling (train only), prefetching. The pipeline lives in `src/data/dataset.py`; here we verify the shift on one example.

In [ ]:
for (src, dec_in), tgt in bundle.train_ds.unbatch().take(1):
    print('tgt   :', tgt.numpy()[:12])
    print('dec_in:', dec_in.numpy()[:12], ' <- shifted right by one (teacher forcing)')
    break

## 10 · Positional encoding
Sinusoidal encodings implemented from the paper: `PE(pos, 2i) = sin(pos / 10000^(2i/d_model))`, `PE(pos, 2i+1) = cos(...)`.

In [ ]:
from src.model.embedding import sinusoidal_position_encoding
from src.visualization.plots import plot_positional_encoding
pe = sinusoidal_position_encoding(cfg.max_seq_len, cfg.d_model)
print('PE matrix shape:', pe.shape)
plot_positional_encoding(pe[np.newaxis, ...], 'outputs/positional_encoding/positional_encoding.png')

## 11 · Scaled dot-product attention from scratch
`softmax(QKᵀ/√d_k)·V`, computed step by step. We inspect scores, weights, and output, and verify softmax rows and masking.

In [ ]:
import tensorflow as tf
from src.model.attention import scaled_dot_product_attention
q = tf.random.normal((2, 4, 5, 16)); k = tf.random.normal((2, 4, 6, 16)); v = tf.random.normal((2, 4, 6, 16))
out, w = scaled_dot_product_attention(q, k, v)
print('output:', out.shape, '| weights:', w.shape)
print('softmax rows sum to 1:', tf.reduce_all(tf.abs(tf.reduce_sum(w, -1) - 1) < 1e-5).numpy())

## 12–13 · Masking system (padding + causal)
Mask convention: **1 = blocked**. Padding masks hide `<PAD>`; the causal (look-ahead) mask stops the decoder from seeing the future; the decoder mask combines both.

In [ ]:
import matplotlib.pyplot as plt
from src.model.masks import create_padding_mask, create_look_ahead_mask, create_decoder_mask
seq = tf.constant([[7, 5, 0, 0]])
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, m, t in [(axes[0], create_padding_mask(seq)[0, 0], 'padding mask (keys)'),
                 (axes[1], create_look_ahead_mask(4), 'causal mask'),
                 (axes[2], create_decoder_mask(seq)[0, 0], 'decoder combined mask')]:
    shown = m.numpy()
    if shown.shape[0] == 1: shown = shown.reshape(-1, shown.shape[-1])
    ax.imshow(shown, cmap='gray_r', vmin=0, vmax=1)
    ax.set_title(t)
plt.tight_layout(); plt.show()

## 14 · Multi-head attention
Manual head split/merge around our attention: `d_model = num_heads × depth`.

In [ ]:
from src.model.layers import MultiHeadAttention
mha = MultiHeadAttention(cfg.d_model, cfg.num_heads)
x = tf.random.normal((2, 10, cfg.d_model))
print('mha out:', mha(x, x, x).shape, '| depth =', cfg.d_model // cfg.num_heads)
print('d_model = heads × depth:', cfg.num_heads * (cfg.d_model // cfg.num_heads) == cfg.d_model)

## 15–18 · Encoder, decoder, complete Transformer
Encoder block: self-attn → Add&Norm → FFN → Add&Norm. Decoder block adds **masked self-attn** (Q=K=V=decoder) and **cross-attn** (Q=decoder, K=V=encoder).

In [ ]:
from src.model.transformer import Transformer, count_parameters
model = Transformer(cfg, len(src_tok), len(tgt_tok))
print(f'parameters: {count_parameters(model):,}')
logits = model((src, dec_in), training=False)
print('logits:', logits.shape, '| finite:', tf.math.is_finite(logits).numpy().all())

## 19–20 · Model sanity checks
Shapes, masks, finite logits, gradient flow — all covered by `tests/` (run `pytest tests/ -v`).

In [ ]:
with tf.GradientTape() as tape:
    logits = model((src, dec_in), training=True)
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=tgt, logits=logits)
grads = tape.gradient(loss, model.trainable_variables)
print('all gradients present:', all(g is not None for g in grads))
print('all gradients finite:', all(tf.math.is_finite(g).numpy().all() for g in grads))

## 21 · Tiny-dataset overfitting test
**Gate:** the model must overfit one batch within ~100 steps. If it cannot, we stop and debug.

In [ ]:
from src.training.trainer import Trainer
trainer = Trainer(model, cfg)  # scratch trainer for the gate
losses = trainer.overfit_check(bundle.train_ds, steps=100)

## 22–23 · Training pipeline + baseline run
Custom `GradientTape` loop, warmup LR schedule, gradient clipping, padding-aware loss, best-val checkpointing. (10 epochs ≈ a few minutes on a T4.)

In [ ]:
model = Transformer(cfg, len(src_tok), len(tgt_tok))  # fresh model for the real run
trainer = Trainer(model, cfg, checkpoint_dir='outputs/model/best_model')
history = trainer.fit(bundle.train_ds, bundle.val_ds, log_every=200)
from src.visualization.plots import plot_training_curves
plot_training_curves(history, 'outputs/training_curves')

## 24 · Autoregressive inference
No teacher forcing now: `<SOS>` → predict → append → predict … → `<EOS>`.

In [ ]:
from src.inference.generate import translate
for i in range(5):
    print(f'EN: {bundle.test_src_text[i]}')
    print(f'FR ref: {bundle.test_tgt_text[i]}')
    print(f'FR hyp: {translate(model, src_tok, tgt_tok, bundle.test_src_text[i])}\n')

## 25 · Test-set evaluation + failure analysis
Held-out test metrics: teacher-forced loss/accuracy + exact match / BLEU on greedy generations, with failure cases harvested to CSV.

In [ ]:
from src.evaluation.evaluate import evaluate_test_set
import json
metrics = evaluate_test_set(model, bundle, 'outputs/predictions/test_predictions.csv', n=500)
test_loss, test_acc = trainer.evaluate(bundle.test_ds)
metrics.update(test_loss=test_loss, test_token_accuracy=test_acc)
print(json.dumps(metrics, indent=2))

## 26 · Attention visualization
Capture attention weights and save heatmaps: encoder self-attn, decoder self-attn (causal), cross-attn.

In [ ]:
from src.visualization.plots import visualize_attentions
paths = visualize_attentions(model, bundle, bundle.test_src_text[0], bundle.test_tgt_text[0],
                             out_dir='outputs/attention_maps', prefix='test_0')
print('\n'.join(paths))

## 27 · Ablation study
Train ~13 configurations (PE on/off, 1/2/4/8 heads, width, depth, FFN, warmup) on the shared dataset; results go to `outputs/experiments/experiment_results.csv`. (Reduced budget: 3 epochs.)

In [ ]:
from src.experiments.ablations import run_experiments
df = run_experiments(cfg, bundle=bundle, epochs=3)
df

## 28 · Built-in benchmark
Identical architecture built from Keras' fused `MultiHeadAttention` — same trainer, same budget.

In [ ]:
from src.experiments.ablations import run_builtin_benchmark
run_builtin_benchmark(cfg, bundle, epochs=3)
import pandas as pd
results = pd.read_csv('outputs/experiments/experiment_results.csv')
results

## 29–31 · Scientific analysis, conclusions, checklist
Questions the table answers: does positional encoding matter? multi-head vs single-head? width vs depth? FFN capacity? warmup schedule? size/quality trade-off? hand-built vs framework MHA? Write your conclusions here — including surprises and remaining failure cases (see `outputs/predictions/test_predictions.csv`).

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(results['params'], results['val_bleu'], s=60)
for _, r in results.iterrows():
    ax.annotate(r['experiment'], (r['params'], r['val_bleu']), fontsize=7, alpha=0.7)
ax.set_xlabel('parameters'); ax.set_ylabel('val BLEU'); ax.set_title('Size / quality trade-off')
plt.tight_layout(); plt.show()

### Definition of done
Dataset · custom tokenizer · tf.data pipeline · positional encoding · scaled dot-product attention · padding/causal masks · multi-head attention · FFN · encoder · decoder · complete Transformer · teacher forcing · padding-aware loss · GradientTape loop · overfit gate · autoregressive inference · test evaluation · attention extraction/visualization · ablations · hyperparameter comparison · built-in benchmark · error analysis · results table · clean notebook · README · reproducible repo.